
# Tests for `position.py` and `portfolio.py`

This notebook gives you a compact, runnable test suite for the current APIs you shared in chat.

## Assumed current API

### `Position`
- `Position(country)`
- `update(trade_qty, notional, initial_margin)`
- `get_notional()`
- `calc_today_pnl(date, new_p)`
- `liquidate()`

### `Portfolio`
- `Portfolio(equity)`
- `update_position(country, trade_qty, notional, initial_margin)`
- `get_margin_used()`
- `get_position(country)`
- `liquidate_position(country)`
- `get_today_pnl(country, date, new_p)`

## What these tests are meant to do
- Validate the **basic state transitions** you want.
- Catch logic bugs early.
- Make it obvious where the implementation still needs work.

Some tests are intentionally strict. If one fails, that usually means either:
1. the implementation is incomplete, or
2. the behavior should be clarified and the test should be adjusted.

Run the cells top to bottom.


In [1]:

# Standard imports
import unittest
from pathlib import Path
import sys
import importlib



## Import project modules

This cell assumes the notebook is inside your repo and that `position.py` / `portfolio.py`
are importable from the project root or the `backtester/` directory.


In [2]:

# Try a few common repo layouts
repo_root = Path.cwd()

candidate_paths = [
    repo_root,
    repo_root / "backtester",
]

for p in candidate_paths:
    if p.exists():
        sys.path.insert(0, str(p))

position_module = importlib.import_module("position")
portfolio_module = importlib.import_module("portfolio")

Position = position_module.Position
Portfolio = portfolio_module.Portfolio

print("Imported Position from:", position_module.__file__)
print("Imported Portfolio from:", portfolio_module.__file__)


Imported Position from: /Users/jonathankim/Documents/GitHub/qts-cross-asset-fx/backtester/position.py
Imported Portfolio from: /Users/jonathankim/Documents/GitHub/qts-cross-asset-fx/backtester/portfolio.py



## Helper notes

These tests are based on the design discussion we just had:

- A brand-new position should have **no PnL** on the first observed price.
- `last_price` should be initialized from the first observed price, not from `0.0`.
- Liquidation should zero out the position.
- Portfolio should create a `Position` automatically when first trading a country.

If your implementation chooses a different convention, update the expected values accordingly.


In [3]:

class TestPositionBasics(unittest.TestCase):
    def setUp(self):
        self.pos = Position("JPY")

    def test_init_country_is_stored(self):
        self.assertEqual(self.pos.get_country(), "JPY")

    def test_initial_state_has_zero_notional(self):
        # Assumes a new position starts empty
        self.assertEqual(self.pos.get_exposure(), 0.0)

    def test_first_price_observation_has_zero_pnl(self):
        # First observed price should initialize state and return 0.0 PnL
        pnl = self.pos.calc_pnl("2026-03-05", 1.20)
        self.assertEqual(pnl, 0.0)

    def test_update_positive_trade_increases_notional(self):
        self.pos.update(trade_qty=1.0, notional=100000.0, initial_margin=10000.0)
        self.assertEqual(self.pos.get_exposure(), 100000.0)

    def test_update_multiple_trades_accumulates_notional(self):
        self.pos.update(trade_qty=1.0, notional=100000.0, initial_margin=10000.0)
        self.pos.update(trade_qty=2.0, notional=250000.0, initial_margin=25000.0)
        self.assertEqual(self.pos.get_exposure(), 350000.0)

    def test_liquidate_resets_notional(self):
        self.pos.update(trade_qty=1.0, notional=100000.0, initial_margin=10000.0)
        self.pos.liquidate()
        self.assertEqual(self.pos.get_exposure(), 0.0)


In [4]:

class TestPositionPnL(unittest.TestCase):
    def setUp(self):
        self.pos = Position("CAD")

    def test_pnl_zero_when_no_previous_price(self):
        self.assertEqual(self.pos.calc_pnl("2026-03-05", 1.10), 0.0)

    def test_long_position_positive_pnl_when_price_rises(self):
        # These tests assume update() establishes a long exposure when trade_qty > 0
        self.pos.update(trade_qty=1.0, notional=100000.0, initial_margin=10000.0)

        # First call initializes last price
        self.pos.calc_pnl("2026-03-05", 1.20)

        # Second call should produce positive pnl if long
        pnl = self.pos.calc_pnl("2026-03-06", 1.25)
        self.assertGreater(pnl, 0.0)

    def test_long_position_negative_pnl_when_price_falls(self):
        self.pos.update(trade_qty=1.0, notional=100000.0, initial_margin=10000.0)
        self.pos.calc_pnl("2026-03-05", 1.20)
        pnl = self.pos.calc_pnl("2026-03-06", 1.15)
        self.assertLess(pnl, 0.0)

    def test_short_position_positive_pnl_when_price_falls(self):
        # Assumes trade_qty < 0 means short exposure
        self.pos.update(trade_qty=-1.0, notional=100000.0, initial_margin=10000.0)
        self.pos.calc_pnl("2026-03-05", 1.20)
        pnl = self.pos.calc_pnl("2026-03-06", 1.15)
        self.assertGreater(pnl, 0.0)

    def test_short_position_negative_pnl_when_price_rises(self):
        self.pos.update(trade_qty=-1.0, notional=100000.0, initial_margin=10000.0)
        self.pos.calc_pnl("2026-03-05", 1.20)
        pnl = self.pos.calc_pnl("2026-03-06", 1.25)
        self.assertLess(pnl, 0.0)



## Portfolio tests

These focus on the behavior your posted `Portfolio` class is supposed to provide.


In [5]:

class TestPortfolioBasics(unittest.TestCase):
    def setUp(self):
        self.portfolio = Portfolio(2_000_000.0)

    def test_init_equity_is_stored(self):
        self.assertEqual(self.portfolio.equity, 2_000_000.0)

    def test_update_position_creates_country_if_missing(self):
        self.portfolio.update_position("JPY", trade_qty=1.0, trade_exposure=100000.0, initial_margin=10000.0)
        self.assertIn("JPY", self.portfolio.positions)

    def test_get_position_returns_position_object(self):
        self.portfolio.update_position("JPY", trade_qty=1.0, trade_exposure=100000.0, initial_margin=10000.0)
        pos = self.portfolio.get_position("JPY")
        self.assertIsInstance(pos, Position)

    def test_get_position_raises_for_missing_country(self):
        with self.assertRaises(ValueError):
            self.portfolio.get_position("DOES_NOT_EXIST")

    def test_liquidate_position_removes_country(self):
        self.portfolio.update_position("JPY", trade_qty=1.0, trade_exposure=100000.0, initial_margin=10000.0)
        self.portfolio.liquidate_position("JPY")
        self.assertNotIn("JPY", self.portfolio.positions)

    def test_liquidate_missing_country_raises(self):
        with self.assertRaises(ValueError):
            self.portfolio.liquidate_position("DOES_NOT_EXIST")


In [6]:

class TestPortfolioMarginAndPnL(unittest.TestCase):
    def setUp(self):
        self.portfolio = Portfolio(2_000_000.0)

    def test_margin_used_single_country(self):
        self.portfolio.update_position("JPY", trade_qty=1.0, trade_exposure=100000.0, initial_margin=10000.0)

        # IMPORTANT:
        # This test reflects the *conceptual* expectation that portfolio margin used
        # should be the sum of initial margins, not the sum of notionals.
        #
        # If your current implementation returns notional instead, this test should fail,
        # which is actually useful because it reveals the bug.
        try:
            margin_used = self.portfolio.get_margin_used()
            self.assertEqual(margin_used, 10000.0)
        except AssertionError:
            print("Current implementation appears to sum notional instead of margin. Fix get_margin_used().")
            raise

    def test_margin_used_multiple_countries(self):
        self.portfolio.update_position("JPY", trade_qty=1.0, trade_exposure=100000.0, initial_margin=10000.0)
        self.portfolio.update_position("CAD", trade_qty=2.0, trade_exposure=250000.0, initial_margin=25000.0)
        try:
            margin_used = self.portfolio.get_margin_used()
            self.assertEqual(margin_used, 35000.0)
        except AssertionError:
            print("Current implementation appears to sum notional instead of margin. Fix get_margin_used().")
            raise

    def test_get_today_pnl_delegates_to_country_position(self):
        self.portfolio.update_position("JPY", trade_qty=1.0, trade_exposure=100000.0, initial_margin=10000.0)

        # first price initializes
        pnl1 = self.portfolio.get_today_pnl("JPY", "2026-03-05", 1.20)
        self.assertEqual(pnl1, 0.0)

        # second price should produce a float
        pnl2 = self.portfolio.get_today_pnl("JPY", "2026-03-06", 1.21)
        self.assertIsInstance(pnl2, float)

    def test_get_today_pnl_missing_country_raises(self):
        with self.assertRaises(ValueError):
            self.portfolio.get_today_pnl("DOES_NOT_EXIST", "2026-03-05", 1.20)



## Run all tests


In [7]:

suite = unittest.TestSuite()

for test_cls in [
    TestPositionBasics,
    TestPositionPnL,
    TestPortfolioBasics,
    TestPortfolioMarginAndPnL,
]:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(test_cls))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


test_first_price_observation_has_zero_pnl (__main__.TestPositionBasics.test_first_price_observation_has_zero_pnl) ... ok
test_init_country_is_stored (__main__.TestPositionBasics.test_init_country_is_stored) ... ok
test_initial_state_has_zero_notional (__main__.TestPositionBasics.test_initial_state_has_zero_notional) ... ok
test_liquidate_resets_notional (__main__.TestPositionBasics.test_liquidate_resets_notional) ... ERROR
test_update_multiple_trades_accumulates_notional (__main__.TestPositionBasics.test_update_multiple_trades_accumulates_notional) ... ok
test_update_positive_trade_increases_notional (__main__.TestPositionBasics.test_update_positive_trade_increases_notional) ... ok
test_long_position_negative_pnl_when_price_falls (__main__.TestPositionPnL.test_long_position_negative_pnl_when_price_falls) ... ok
test_long_position_positive_pnl_when_price_rises (__main__.TestPositionPnL.test_long_position_positive_pnl_when_price_rises) ... ok
test_pnl_zero_when_no_previous_price (__main_

<unittest.runner.TextTestResult run=21 errors=1 failures=0>


## What failures probably mean

### If `test_first_price_observation_has_zero_pnl` fails
Your `Position.calc_today_pnl()` is probably using `last_price = 0.0` instead of initializing from the first seen price.

### If `test_margin_used_*` fails
Your `Portfolio.get_margin_used()` is probably summing **notional** instead of **margin**.

### If long/short PnL sign tests fail
Your position sign convention is inconsistent somewhere in:
- `update()`
- `calc_today_pnl()`

### If `liquidate` tests fail
Your liquidation logic is not resetting/removing state fully.
